![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Lab: RAG from scratch with the RedisVL
---

## Introduction

In this lab, you'll build a RAG chatbot that can answer questions about Nike’s 10-K financial filing. You'll work with unstructured data (PDF) and also utilize an LLM inference engine (OpenAI) to build a complete pipeline that can respond to natural language queries about the financial data. 

As previously mentioned, this lab is broken out into six sections:

1. Introduction (📍 You are here )
2. Define Redis connection settings and embed data
3. Practice vector search
4. Build a RAG Pipeline
5. Run the RAG chatbot
6. Wrap Up

This lab will also include a few tasks (denoted with a 📌) for you to work with the code. When applicable, tasks come with a solution code to compare.

Additionally, if you ever want to view the database at any point during the lab, you can return to the lab documentation page and open Redis Insight.

Now, let's get started.

## Define Redis connection settings and embed data

Before we build our RAG pipeline, we'll need two things: Redis connection properties (to connect to Redis) and some data to work with.

For this lab, we’ll work with a real-world document: Nike’s 10-K financial filing. This PDF contains detailed business, financial, and risk information that we'll extract and prepare for search. This file can be found by using the file browser and navigating inside the `/resources-rag` directory.

Our goal in this section is to load the PDF, chunk it into passages, embed those chunks as vectors, store them in Redis, and index them for search.

Let's first start by setting up the connection details.

### Setup Redis connection variables

Below is the code that sets up the necessary connection variables for connecting to Redis. Run the code block.

In [1]:
import os

REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", "6379")
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

REDIS_URL = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"
os.environ["REDIS_URL"] = REDIS_URL

print("✅ Redis connection variables set.")

✅ Redis connection variables set.


### Chunking the data

In this lab, we’re working with a single unstructured PDF (Nike’s 10-K filing), which contains long passages of text. Embedding this type of document presents two key issues:

- Token limits: Most embedding models can’t handle extremely long inputs.
- Granularity: Embedding the full document would make it hard to retrieve specific, relevant passages.

To address this, we’ll break the document into smaller chunks that can be embedded and searched independently.

Rather than writing our own PDF parsing and chunking logic, we'll use two utilities from [LangChain](https://github.com/langchain-ai/langchain), an open-source framework for LLM apps:

- `PyPDFLoader`: This class will read and extract text from our PDF file. The class requires one main argument called `file_path`, which specifies the path to the PDF file you want to load.

- `RecursiveCharacterTextSplitter`: This class will break long text into chunks, preserving context across chunk boundaries. We'll provide two arguments:

    - `chunk_size`: Specifies the characters per chunk

    - `chunk_overlap`: Specifies how much overlap between the chunks we want.

📌 **Task 1: Set up the text splitter and loader**  

1. Create a variable named `loader` by calling the `PyPDFLoader` class with the `doc` variable as the file path. 

2. Create a new variable called `text_splitter` by instantiating the `RecursiveCharacterTextSplitter` class. Set the `chunk_size` parameter to `2500` and the `chunk_overlap` parameter to `0`.
    
The solution for this task is posted below. To avoid issues, make sure to compare the code you write against the solution code and then run the code block.

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# Path to the Nike 10-K PDF directly
doc = "resources-rag/nke-10k-2023.pdf"

# --- Define text splitter and loader below ----
loader = PyPDFLoader(doc)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2500,
    chunk_overlap = 0
)
    
# Confirms the text splitter was created successfully
print("Text splitter object:", text_splitter)

# Confirms the loader was created successfully
print("Loader: object", loader)

Text splitter object: <langchain_text_splitters.character.RecursiveCharacterTextSplitter object at 0x7faf557e3450>
Loader: object <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x7faf557e34d0>


<details>
<summary> 🔒 Loader and Text Splitter Solution Code </summary>
<br>

```python

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# Specify the path to the Nike 10-K PDF directly
doc = "resources-rag/nke-10k-2023.pdf"

# Add the text splitter and loader below
loader = PyPDFLoader(doc)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,  # characters per chunk
    chunk_overlap=0   # no overlap between chunks
)

# Confirms the text splitter was created successfully
print("Text Splitter:", text_splitter)

# Confirms the loader was created successfully
print("Loader:", loader)

```

</details>

Next, we can use the loader's `.load_and_split()` method to create the chunks. There will be approximately 221 chunks created. Review the code below, run it, and observe the results.

⚠️ **Note:** The code block below may take some time to run. Please be patient as it completes, and observe the results.

In [4]:
# Extract, load, and split the document into chunks
chunks = loader.load_and_split(text_splitter)

print("Done preprocessing. Created", len(chunks), "chunks of the original PDF:", doc)

Done preprocessing. Created 211 chunks of the original PDF: resources-rag/nke-10k-2023.pdf


### Text embedding generation with RedisVL

It’s time to to embed the data! In this lab, you will create the vectorizer from scratch. 

📌 **Task 2: Create the vectorizer** 

Create a new variable called `hf` by initializing the `HFTextVectorizer` class. Set it up with the following properties:

- Use the model `"sentence-transformers/all-MiniLM-L6-v2"`
- Assign it an instance of `EmbeddingsCache` via the `cache` parameter with the following properties:
    - `name="embedcache"`
    - `ttl=600`
    - `redis_url=REDIS_URL`

The solution is posted below the code block. Make sure to confirm your solution and then run the code block.

⚠️ **Note:** The code block below may take some time to run. Please be patient as it completes, and observe the results.

In [5]:
import warnings
import pandas as pd
from redisvl.utils.vectorize import HFTextVectorizer, BaseVectorizer
from redisvl.extensions.cache.embeddings import EmbeddingsCache

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Create the new vectorizer below
hf = HFTextVectorizer(
    model = "sentence-transformers/all-MiniLM-L6-v2",
    cache = EmbeddingsCache(
        name = "embedcache",
        ttl = 600,
        redis_url = REDIS_URL,
    )
)

02:52:23 sentence_transformers.SentenceTransformer INFO   Use pytorch device_name: cpu
02:52:23 sentence_transformers.SentenceTransformer INFO   Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

<details>
<summary> 🔒 Vectorizer Solution Code </summary>
<br>

```python

hf = HFTextVectorizer(
    model="sentence-transformers/all-MiniLM-L6-v2",
    cache=EmbeddingsCache(
        name="embedcache",
        ttl=600,
        redis_url=REDIS_URL,
    )
)

```

</details>

Next, we'll create the embeddings using the vectorizer (`hf`). Review the code below, run it, and observe the results.

⚠️ **Note:** The code block below may take some time to run. Please be patient as it completes, and observe the results.

In [6]:
# Embed each chunk content
embeddings = hf.embed_many([chunk.page_content for chunk in chunks])

# Check to make sure we've created enough embeddings, 1 per document chunk
len(embeddings) == len(chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

True

### Define the index schema

Now, we'll need to create an index schema. You will handle writing this code in the task below.

📌 **Task 3: Create an index schema**  

Create a new variable (for a Python dictionary) called `schema` that defines the structure of the index for storing your document chunks in Redis. Your schema should include:

Create a schema named `"redisvl"` (you can plug in the variable `index_name` we provided) with:
- A prefix of `"chunk"`
- A storage type of `"hash"`

Include the following fields:

| Field Name   | Type      | Attributes                                                                 |
|--------------|-----------|---------------------------------------------------------------------------|
| `chunk_id`      | `tag`    | `sortable: True`                                                                   |
| `content`| `text`    | _none_                                                                     |
| `text_embedding`     | `vector`  | `dims: 384`  <br> `distance_metric: "cosine"` <br> `algorithm: "hnsw"` <br> `datatype: "float32"` |


The solution is posted below. Make sure to confirm your solution and then run the code.

In [8]:
from redisvl.index import SearchIndex

index_name = "redisvl"

schema = {
  "index": {
    "name": index_name,
    "prefix": "chunk",
    "storage_type": "hash"
  },
  "fields": [
    {
        "name": "chunk_id",
        "type": "tag",
        "attrs": {
            "sortable": True
        }
    },
    {
        "name": "content",
        "type": "text"
    },
    {
        "name": "text_embedding",
        "type": "vector",
        "attrs": {
            "dims": 384,
            "distance_metric": "cosine",
            "algorithm": "hnsw",
            "datatype": "float32"
        }
    }
  ]
}


print("Schema created successfully:", schema)

Schema created successfully: {'index': {'name': 'redisvl', 'prefix': 'chunk', 'storage_type': 'hash'}, 'fields': [{'name': 'chunk_id', 'type': 'tag', 'attrs': {'sortable': True}}, {'name': 'content', 'type': 'text'}, {'name': 'text_embedding', 'type': 'vector', 'attrs': {'dims': 384, 'distance_metric': 'cosine', 'algorithm': 'hnsw', 'datatype': 'float32'}}]}


<details>
<summary> 🔒 Schema Solution Code </summary>
<br>

```python

from redisvl.index import SearchIndex

index_name = "redisvl"

schema = {
  "index": {
    "name": index_name,
    "prefix": "chunk",
    "storage_type": "hash"
  },
  "fields": [
    {
        "name": "chunk_id",
        "type": "tag",
        "attrs": {
            "sortable": True
        }
    },
    {
        "name": "content",
        "type": "text"
    },
    {
        "name": "text_embedding",
        "type": "vector",
        "attrs": {
            "dims": 384,
            "distance_metric": "cosine",
            "algorithm": "hnsw",
            "datatype": "float32"
        }
    }
  ]
}

print("Schema created successfully:", schema)
```

</details>

### Index creation

With the schema created, we can now create the index and load it into Redis. Run the code block.

In [9]:
# create an index from schema and the client
index = SearchIndex.from_dict(schema, redis_url=REDIS_URL)
index.create(overwrite=True, drop=True)

print("Index created successfully:", index)

Index created successfully: <redisvl.index.index.SearchIndex object at 0x7fae1427aa90>


We can additionally confirm the index exists with the following RedisVL CLI commands. Run the code block to view the index inside Redis.

In [10]:
# List all indices. 
!rvl index listall

# Get info about the index
!rvl index info -i redisvl

03:12:07 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL
03:12:07 [RedisVL] INFO   Indices:
03:12:07 [RedisVL] INFO   1. redisvl
03:12:08 [RedisVL] INFO   Using Redis address from environment variable, REDIS_URL


Index Information:
╭───────────────┬───────────────┬───────────────┬───────────────┬───────────────╮
│ Index Name    │ Storage Type  │ Prefixes      │ Index Options │ Indexing      │
├───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
| redisvl       | HASH          | ['chunk']     | []            | 0             |
╰───────────────┴───────────────┴───────────────┴───────────────┴───────────────╯
Index Fields:
╭─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────╮
│ Name            │ Attribute       │ Type   

### Create and load the dataset

It’s time to store the data in Redis so we can start searching it. We will need to manually assemble a list of Python dictionaries (one per chunk) so the structure matches the schema. 

Each entry should have the following properties:

- chunk_id: a unique ID for the Redis key
- content: the chunk’s text content
- text_embedding: the vector embedding, converted to a byte buffer using `array_to_buffer()`

Examine the code below for how we accomplish this, and then run the code block.

In [11]:
from redisvl.redis.utils import array_to_buffer

data = [
    {
        'chunk_id': i,
        'content': chunk.page_content,
        # For HASH -- we must convert embeddings to bytes
        'text_embedding': array_to_buffer(embeddings[i], dtype='float32')
    } for i, chunk in enumerate(chunks)
]

index.load(data, id_field="chunk_id")

['chunk:0',
 'chunk:1',
 'chunk:2',
 'chunk:3',
 'chunk:4',
 'chunk:5',
 'chunk:6',
 'chunk:7',
 'chunk:8',
 'chunk:9',
 'chunk:10',
 'chunk:11',
 'chunk:12',
 'chunk:13',
 'chunk:14',
 'chunk:15',
 'chunk:16',
 'chunk:17',
 'chunk:18',
 'chunk:19',
 'chunk:20',
 'chunk:21',
 'chunk:22',
 'chunk:23',
 'chunk:24',
 'chunk:25',
 'chunk:26',
 'chunk:27',
 'chunk:28',
 'chunk:29',
 'chunk:30',
 'chunk:31',
 'chunk:32',
 'chunk:33',
 'chunk:34',
 'chunk:35',
 'chunk:36',
 'chunk:37',
 'chunk:38',
 'chunk:39',
 'chunk:40',
 'chunk:41',
 'chunk:42',
 'chunk:43',
 'chunk:44',
 'chunk:45',
 'chunk:46',
 'chunk:47',
 'chunk:48',
 'chunk:49',
 'chunk:50',
 'chunk:51',
 'chunk:52',
 'chunk:53',
 'chunk:54',
 'chunk:55',
 'chunk:56',
 'chunk:57',
 'chunk:58',
 'chunk:59',
 'chunk:60',
 'chunk:61',
 'chunk:62',
 'chunk:63',
 'chunk:64',
 'chunk:65',
 'chunk:66',
 'chunk:67',
 'chunk:68',
 'chunk:69',
 'chunk:70',
 'chunk:71',
 'chunk:72',
 'chunk:73',
 'chunk:74',
 'chunk:75',
 'chunk:76',
 'chunk:7

<details>
<summary> (Optional) Detailed code breakdown</summary>
<br>

Here’s what we are doing:

1. We loop through each chunk and its corresponding vector in embeddings.

2. For each chunk, we create a dictionary with:

    - chunk_id: a unique ID based on its index in the list (this will be used as the Redis key)

    - content: the actual text content of the chunk

    - text_embedding: the vector for that chunk, converted to a binary buffer using `array_to_buffer()` (this is required for Redis hashes)

The result is a list of dictionaries — one per chunk — with each associated with a key that follows the pattern defined by the schema (e.g., `chunk:80`)
</details>

## Verify the setup with vector search

We're now all set to begin running searches. Before jumping into the RAG portion of this lab, let's take a moment to verify that vector search is working.

📌 **Task: KNN Search**  

Let's implement a KNN search. Write a KNN search to find the top 5 most relevant chunks from the Nike 10-K PDF based on the query `"What were Nike's profit margins and company performance?"`. 

A few things to note as you write your code: 

1. Remember to correctly import the appropriate query class from `redisvl.query`.

2. The query object should:
    - Have return fields for `"chunk_id"` and `"content"`
    - A vector field name of `"text_embedding"`
    - Return the score

In [12]:
from redisvl.query import VectorQuery

query = "What were Nike's profit margins and company performance"

query_embedding = hf.embed(query)

vector_query = VectorQuery(
    vector=query_embedding,
    vector_field_name="text_embedding",
    num_results=5,
    return_fields=["chunk_id", "content"],
    return_score=True
)

# execute the query with RedisVL
result=index.query(vector_query)

# view the results
pd.DataFrame(result)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,id,vector_distance,chunk_id,content
0,chunk:80,0.311066567898,80,Table of Contents\nCONSOLIDATED OPERATING RESU...
1,chunk:88,0.321317493916,88,"Asia Pacific & Latin America 1,932 1,896 2 % 1..."
2,chunk:87,0.340714752674,87,Table of Contents\nOPERATING SEGMENTS\nAs disc...
3,chunk:82,0.358113706112,82,Table of Contents\nFISCAL 2023 NIKE BRAND REVE...
4,chunk:27,0.359879732132,27,"• In the past, certain retailers of our produc..."


<details>
<summary> 🔒 KNN Search Solution Code </summary>
<br>

```python

from redisvl.query import VectorQuery

query = "What were Nike's profit margins and company performance"

query_embedding = hf.embed(query)

vector_query = VectorQuery(
    vector=query_embedding,
    vector_field_name="text_embedding",
    num_results=5,
    return_fields=["chunk_id", "content"],
    return_score=True
)

# execute the query with RedisVL
result=index.query(vector_query)

# view the results
pd.DataFrame(result)

```

</details>

## Build a RAG Pipeline 

Notice that when we performed KNN search above to retrieve relevant chunks from Redis, the chunks themselves weren't very useful in generating a meaningful response to the query. The real power comes when we pass them as context to an LLM to answer the query.

In this section, we'll write the logic to accept a user question, search Redis for relevant context, and use that context to generate a response with a language model.

### Set up RedisVL AsyncSearchIndex

So far, we have used `SearchIndex` from RedisVL to create and query an index. When used, this class was performing operations in a synchronous (blocking) manner. That was perfectly fine for our exploratory code, but real-world applications — especially anything with a UI or concurrent workloads - benefit from non-blocking operations. 

The good news is that RedisVL also offers an `AsyncSearchIndex` class. This class lets us query Redis using Python’s `async` and `await` syntax.

Below, we create the new index using the class, reusing the same schema from earlier. Examine the code below and then run it.

In [13]:
from redisvl.index import AsyncSearchIndex

async_index = AsyncSearchIndex.from_dict(schema, redis_url=REDIS_URL)

print("Async Index created successfully:", async_index)

Async Index created successfully: <redisvl.index.index.AsyncSearchIndex object at 0x7fad90327110>


Next, let's work on setting up the LLM (OpenAI).

### Set up OpenAI API

To generate answers from the retrieved Redis context, we’ll need to connect to a large language model API. In this lab, we’ll use the `gpt-4o-mini` model via OpenAI’s Python client library. We have already set up an OpenAI key and the base API URL for you.

Run the code block below to set the environment variables.

In [14]:
import openai
import os
import getpass

CHAT_MODEL = "gpt-4.1"

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY :")

BASE_URL = os.environ["OPENAI_API_BASE"]

print("✅ OpenAI connection variables set successfully.")

✅ OpenAI connection variables set successfully.


### Baseline Retrieval Augmented Generation

Now that our document chunks are embedded and indexed in Redis, it’s time to build the RAG pipeline to power a chat experience.

The workflow will have four key steps:

1. Capture the question – In this lab, in the "Test it out" section below, you will be using a prebuilt small UI chat interface to simulate the chat experience of submitting a question.

2. Embed the question – Just like our chunks, the query must be embedded for semantic search.

3. Retrieve context from Redis – Use KNN search to find the most relevant chunks for the query.

4. Generate the answer – Construct a prompt with the retrieved context and embedded question, then call the LLM and receive a response.

Each of the next sections implements one of these steps. Let’s get started.

#### Prep the system

Before asking questions, it’s important to give the LLM context about its role. This is typically done using a system prompt. You’ve likely seen this before when setting instructions or defining a persona for the model.

For our financial data, we can use the system prompt below. Run the code block to create the variable (we'll use it later). The code block will run silently.

In [15]:
SYSTEM_PROMPT = """You are a helpful financial analyst assistant that has access
to public financial 10k documents in order to answer user's questions about company
performance, ethics, characteristics, and core information."""

#### Query embedding

Next, to search semantically, we need to embed the user’s query into the same vector space as our document chunks. We'll define a helper function called `embed_query` that takes in a query argument and does this embedding for us using the vectorizer (`hf`) from earlier. We'll use this function later when we process the query. 

Examine and then run the code block below to create the helper function. 

The code block will run silently.

In [16]:
def embed_query(query: str):
    """Convert a user query into a dense vector representation."""
    return hf.embed(query)

#### Retrieve relevant chunks from Redis

In addition to handling the embedding for a user's question, we'll need to search Redis for semantically similar content (the "context"). To do that, we’ll define a helper function called `retrieve_context()`.

This function will take in:

- An instance of `AsyncSearchIndex`. We'll eventually pass it the `async_index` we created in earlier steps.

- The embedded user query. We'll eventually pass it the result of the `embed_query` function we defined in the last step, once we have a question to process.

The function returns a combined string of the top matching chunks. This serves as the retrieved context in our Retrieval-Augmented Generation (RAG) pipeline — the information the LLM will reference when generating an answer.

Examine and run the code block below.

The code block will run silently.

In [17]:
async def retrieve_context(async_index: AsyncSearchIndex, query_vector) -> str:
    """Fetch the relevant context from Redis using vector search"""
    
    results = await async_index.query(
        VectorQuery(
            vector=query_vector,
            vector_field_name="text_embedding",
            return_fields=["content"],
            num_results=3
        )
    )

    retrieved_context = "\n".join([result["content"] for result in results])
    return retrieved_context

#### Constructing the prompt

Before we can call the LLM, we'll just need one more helper function to combine the user query and the retrieved context. We'll define a helper function called `promptify()` to do just that. 

Examine and then run the code block below. The code block will run silently.

In [18]:
def promptify(query: str, context: str) -> str:
    return f'''Use the provided context below derived from public financial
    documents to answer the user's question. If you can't answer the user's
    question, based on the context; do not guess. If there is no context at all,
    respond with "I don't have enough context to answer this question.".

    User question:

    {query}

    Helpful context:

    {context}

    Answer:
    '''

#### LLM Response

The final part of our RAG flow is to generate an answer using the LLM. In an earlier step, we configured our OpenAI API credentials and imported the `openai` module. The module provides a class called `AsyncClient` to send the query and context to a model like gpt-4o-mini and return the response.

To make this clean and reusable, we’ll define a helper function called `generate_llm_response()`. Examine the code block below and then run it. A more detailed breakdown under the code block.

In [19]:
async def generate_llm_response(query: str, context: str) -> str:
    """Construct and send a complete prompt to the OpenAI LLM and return its response."""

    # Send prompt to the LLM via OpenAI’s async client
    response = await openai.AsyncClient(base_url=BASE_URL).chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": promptify(query, context)}
        ],
        temperature=0.1,
        seed=42
    )

    # Return just the response content
    return response.choices[0].message.content

<details>
<summary>(Optional) Detailed code breakdown</summary>

Our new function takes in the query and context (we need those for the LLM), and then we call:

```python

await openai.AsyncClient(...).chat.completions.create(...)

```

To break it down further: 

1. First, we use `AsyncClient` and pass it the `BASE_URL` we defined earlier for the OpenAI API. It would look something like `"https://api.openai.com/v1"`.

2. Then, our `.chat.completions.create(...)` calls the `/v1/chat/completions` endpoint, which is used for generating chat-style responses. The `.create()` method can configure a variety of parameters, but most importantly, we want to provide it a model, the messages, and some settings to tweak its response (`temperature` and `seed`). More information on all the methods can be found on the [Python library API documentation](https://github.com/openai/openai-python/blob/main/api.md#completions-1).

3. Finally, we return the content from the first generated message:

    ```python

    return response.choices[0].message.content

    ```

    The call to `.chat.completions.create(...)` returns a response object containing one or more choices, each representing a different possible completion (e.g., if you have ever gotten a prompt to choose one response or another in ChatGPT, this is basically the same). In our case, we’re only asking for a single completion, so we access the first (and only) result with `response.choices[0]`.

    Inside each choice is a message, which has a role (either system, user, or assistant) and content — the actual text generated by the model. So when we call `.message.content`, we’re extracting just the final response text from the assistant.

    This value — a plain string — is what our application will show back to the user in the chatbot interface, making it the final output of the entire RAG pipeline.


</details>
















<details>
<summary>(Optional) What does temperature and seed do?</summary>

When working with LLMs, temperature and seed are commonly used parameters. 

Temperature controls output creativity by adjusting the randomness of token selection: lower values (e.g., 0.2) make the output more deterministic and focused, while higher values (e.g., 0.8 to 2.0) increase randomness and creativity.

The seed parameter provides a way to achieve reproducible output by setting the initial state for the random number generator, meaning the same seed with the same parameters will produce the same result. You can learn more about this parameter by <a href="https://cookbook.openai.com/examples/reproducible_outputs_with_the_seed_parameter"> visiting the feature release post by OpenAI. </a>

</details>

### Bring it all together

Now we have everything we need to bring the RAG call together. While it may seem like a lot, remember it breaks down to three essential steps: 

1. Capture and embed the user query
2. Retrieve the context
3. Call the LLM with both

Here, we'll define our final function, `answer_question`, which calls all our helper functions. Examine and then run the code below. The code block will run silently.

In [20]:
async def answer_question(index: AsyncSearchIndex, query: str):
    """End-to-end RAG: embeds query, retrieves context, generates LLM response"""

    # Step 1: Embed the query
    query_vector = embed_query(query)

    # Step 2: Retrieve matching context chunks from Redis
    context = await retrieve_context(index, query_vector)

    # Step 3: Generate response from OpenAI using system prompt + context
    return await generate_llm_response(query, context)

### Testing the RAG

Below, you will find pre-defined code that will boot up a UI chat interface. Don't worry about the specifics of the code that creates the UI, but if you're curious, it's located in the `chat_ui.py` file in the `/resources-rag` folder.

Under the hood, it uses the `answer_question()` function we defined earlier. Run the code block to boot up the chat UI and then submit any of the following queries (or your own):

- What is the trend in the company's revenue and profit over the past few years?
- What are the company's primary revenue sources?
- How much debt does the company have, and what are its capital expenditure plans?
- What does the company say about its environmental, social, and governance (ESG) practices?

⚠️ **Note:** It may take a moment for the LLM to retrieve a response. Please be patient.

In [21]:
import sys
sys.path.append('resources-rag')

from chat_interface import start_chat

start_chat(answer_question, async_index)

## Wrap Up 🏁 

Great job! You've completed the lab on building a RAG from scratch with RedisVL.

You successfully built a Retrieval-Augmented Generation (RAG) pipeline using RedisVL, LangChain, and OpenAI — all from the ground up.

More specifically, you learned how to:

- Embed and index document chunks in Redis for fast semantic search
- Retrieve relevant context using vector search (the **R** in RAG)
- Augment user queries with that context to generate grounded answers (the **A** in RAG)
- Generate final responses using an LLM (the **G** in RAG)

You now have the foundational skills to build production-ready RAG applications on your own data. In the next lab (Production RAG), you'll productionize the RAG you built with key features like LLM memory and semantic caching. 

Feel free to experiment with different documents, chunking strategies, or LLM prompts to see how the results change. Then, move onto the next lab.